In [21]:
import pandas as pd 
from sklearn.utils import resample
import numpy as np 
from sklearn.preprocessing import OneHotEncoder

In [2]:
Data = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn-selected-columns.csv")
Data

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No
...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No


In [3]:
Data.info()
Data.describe()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   customerID       7043 non-null   str  
 1   gender           7043 non-null   str  
 2   SeniorCitizen    7043 non-null   int64
 3   Partner          7043 non-null   str  
 4   Dependents       7043 non-null   str  
 5   tenure           7043 non-null   int64
 6   PhoneService     7043 non-null   str  
 7   MultipleLines    7043 non-null   str  
 8   InternetService  7043 non-null   str  
 9   OnlineSecurity   7043 non-null   str  
dtypes: int64(2), str(8)
memory usage: 550.4 KB


,SeniorCitizen,tenure
count,7043.000000,7043.000000
mean,0.162147,32.371149
std,0.368612,24.559481
min,0.000000,0.000000
25%,0.000000,9.000000
50%,0.000000,29.000000
75%,0.000000,55.000000
max,1.000000,72.000000


In [5]:
Data.isnull().sum()
# So the Data Doesn't have Missing Values 


customerID         0
gender             0
SeniorCitizen      0
Partner            0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
OnlineSecurity     0
dtype: int64

In [ ]:
Data.drop("Dependents",axis=1,inplace=True)
Data.drop("Partner",axis=1,inplace=True)
Data.drop("SeniorCitizen",axis=1,inplace=True)

In [12]:
Data["Services"] = Data['PhoneService']+"_"+Data['InternetService']


In [ ]:
Data.drop("PhoneService",axis=1,inplace=True)
Data.drop("InternetService",axis=1,inplace=True)

In [27]:
# Encoding the Gender 
Encoder = OneHotEncoder(sparse_output=False)
Encoded_Data = Encoder.fit_transform(Data[['gender']])
Encoded_DataFrame = pd.DataFrame(Encoded_Data,columns=Encoder.get_feature_names_out()).reset_index(drop=True)

Data =pd.concat([Data,Encoded_DataFrame],axis=1).reset_index(drop=True)

In [29]:
Data.drop("gender",axis=1,inplace=True)

In [38]:
Enocder_OS = OneHotEncoder(sparse_output=False)
Encoded_DataOS = Enocder_OS.fit_transform(Data[['OnlineSecurity']])
Encoded_DataF = pd.DataFrame(Encoded_DataOS,columns=Enocder_OS.get_feature_names_out()).reset_index(drop=True)
Data = pd.concat([Data,Encoded_DataF],axis=1)

In [40]:
Data.drop("OnlineSecurity",axis=1,inplace=True)

In [41]:
Data

,customerID,tenure,MultipleLines,Services,gender_Female,gender_Male,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes
0,7590-VHVEG,1,No phone service,No_DSL,1.0,0.0,1.0,0.0,0.0
1,5575-GNVDE,34,No,Yes_DSL,0.0,1.0,0.0,0.0,1.0
2,3668-QPYBK,2,No,Yes_DSL,0.0,1.0,0.0,0.0,1.0
3,7795-CFOCW,45,No phone service,No_DSL,0.0,1.0,0.0,0.0,1.0
4,9237-HQITU,2,No,Yes_Fiber optic,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,24,Yes,Yes_DSL,0.0,1.0,0.0,0.0,1.0
7039,2234-XADUH,72,Yes,Yes_Fiber optic,1.0,0.0,1.0,0.0,0.0
7040,4801-JZAZL,11,No phone service,No_DSL,1.0,0.0,0.0,0.0,1.0
7041,8361-LTMKD,4,Yes,Yes_Fiber optic,0.0,1.0,1.0,0.0,0.0


In [42]:
Data['MultipleLines'].value_counts()

MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

In [ ]:
conacting_Terms=[]

n= 3500

Unique_Class = Data ['MultipleLines'].unique()

for Class in Unique_Class :
    class_data = Data[Data['MultipleLines']==Class]
    Current_Count = len(class_data)

    if Current_Count < n :
        # Upsampliing the data 
        resampled_subset = resample(
            class_data,
            replace=True,      # Mandatory for upsampling
            n_samples=n,
            random_state=50
        )


    conacting_Terms.append(resampled_subset)

Balanced_data = pd.concat(conacting_Terms).reset_index(drop=True)


In [46]:
Balanced_data['MultipleLines'].value_counts()

MultipleLines
No phone service    3500
No                  3500
Yes                 3500
Name: count, dtype: int64

In [52]:
# Let's Do the OneHot Encoding 
encoder = OneHotEncoder(sparse_output=False)
encoded_data = encoder.fit_transform(Balanced_data[['MultipleLines']])

Data_Frame = pd.DataFrame(encoded_data,columns=encoder.get_feature_names_out()).reset_index(drop=True)

Balanced_data = pd.concat([Data_Frame,Balanced_data],axis=1).reset_index(drop=True)

In [53]:
Balanced_data

,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,0,1,2,customerID,tenure,MultipleLines,Services,gender_Female,gender_Male,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes
0,0.0,1.0,0.0,0.0,1.0,0.0,0361-HJRDX,68,No phone service,No_DSL,1.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,1.0,0.0,6791-YBNAK,18,No phone service,No_DSL,0.0,1.0,1.0,0.0,0.0
2,0.0,1.0,0.0,0.0,1.0,0.0,1814-WFGVS,72,No phone service,No_DSL,0.0,1.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0,1.0,0.0,6281-FKEWS,44,No phone service,No_DSL,1.0,0.0,1.0,0.0,0.0
4,0.0,1.0,0.0,0.0,1.0,0.0,7595-EHCDL,32,No phone service,No_DSL,0.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10495,0.0,0.0,1.0,0.0,0.0,1.0,7969-AULMZ,21,Yes,Yes_Fiber optic,1.0,0.0,1.0,0.0,0.0
10496,0.0,0.0,1.0,0.0,0.0,1.0,8984-HPEMB,71,Yes,Yes_Fiber optic,1.0,0.0,0.0,0.0,1.0
10497,0.0,0.0,1.0,0.0,0.0,1.0,0917-EZOLA,72,Yes,Yes_Fiber optic,0.0,1.0,1.0,0.0,0.0
10498,0.0,0.0,1.0,0.0,0.0,1.0,7817-BOQPW,2,Yes,Yes_Fiber optic,1.0,0.0,1.0,0.0,0.0


In [54]:
Balanced_data.drop(0,axis=1,inplace=True)
Balanced_data.drop(1,axis=1,inplace=True)
Balanced_data.drop(2,axis=1,inplace=True)

In [56]:
Balanced_data.drop("MultipleLines",axis=1,inplace=True)

In [58]:
Balanced_data = Balanced_data[
    [
        'customerID',
        'tenure',
        'Services',

        'gender_Female',
        'gender_Male',

        'MultipleLines_No',
        'MultipleLines_No phone service',
        'MultipleLines_Yes',

        'OnlineSecurity_No',
        'OnlineSecurity_No internet service',
        'OnlineSecurity_Yes'
    ]
]

In [60]:
Balanced_data.drop("gender_Female",axis=1,inplace=True)
Balanced_data.drop("customerID",axis=1 ,inplace=True)


In [61]:
Balanced_data

,tenure,Services,gender_Male,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes
0,68,No_DSL,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,18,No_DSL,1.0,0.0,1.0,0.0,1.0,0.0,0.0
2,72,No_DSL,1.0,0.0,1.0,0.0,0.0,0.0,1.0
3,44,No_DSL,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,32,No_DSL,1.0,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
10495,21,Yes_Fiber optic,0.0,0.0,0.0,1.0,1.0,0.0,0.0
10496,71,Yes_Fiber optic,0.0,0.0,0.0,1.0,0.0,0.0,1.0
10497,72,Yes_Fiber optic,1.0,0.0,0.0,1.0,1.0,0.0,0.0
10498,2,Yes_Fiber optic,0.0,0.0,0.0,1.0,1.0,0.0,0.0


In [62]:
Balanced_data['Services'].value_counts()

Services
No_DSL             3500
Yes_Fiber optic    3455
Yes_DSL            1906
Yes_No             1639
Name: count, dtype: int64

In [64]:
Balanced_data['Services'] = Data['Services'].replace({
    'No_DSL': 'NoPhone_DSL',
    'Yes_DSL': 'Phone_DSL',
    'Yes_Fiber optic': 'Phone_Fiber',
    'Yes_No': 'PhoneOnly'
})

In [ ]:
Balanced_data.drop("MultipleLines_Yes",axis=1,inplace=True)      # Droping this Columns Because We care 
Balanced_data.drop("OnlineSecurity_Yes",axis = 1,inplace = True) # about The Customers Are Leaving While Analysis


In [67]:
Balanced_data

,tenure,Services,gender_Male,MultipleLines_No,MultipleLines_No phone service,OnlineSecurity_No,OnlineSecurity_No internet service
0,68,NoPhone_DSL,0.0,0.0,1.0,0.0,0.0
1,18,Phone_DSL,1.0,0.0,1.0,1.0,0.0
2,72,Phone_DSL,1.0,0.0,1.0,0.0,0.0
3,44,NoPhone_DSL,0.0,0.0,1.0,1.0,0.0
4,32,Phone_Fiber,1.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...
10495,21,NaN,0.0,0.0,0.0,1.0,0.0
10496,71,NaN,0.0,0.0,0.0,0.0,0.0
10497,72,NaN,1.0,0.0,0.0,1.0,0.0
10498,2,NaN,0.0,0.0,0.0,1.0,0.0


In [68]:
Balanced_data.to_csv("Transformed_data.csv")